# Functional Data Structures

Collections are one of the most used data structures in functional programming, and where we can immediately see the advantages of using it.

### Seq

In Scala, the sequence Seq is one of the most common data structures, representing an ordered sequence of elements. It can be mutable or immutable.

The Seq `apply` method will create by default a concrete implementation of a sequence, which is a `List`.


In [1]:
Seq.apply(34,56,78)

res1: Seq[Int] = List(34, 56, 78)

As usual the apply method is implied by using `()`:

In [2]:
val seq1 = Seq(4,8,7)

seq1: Seq[Int] = List(4, 8, 7)

Sequences can also be built using the prepend methods. Notice the `Nil` element which is in fact the empty sequence.

In [3]:
val seq2 = 3 +: 4 +: 5 +: Nil

seq2: List[Int] = List(3, 4, 5)

In [4]:
seq2.head

res4: Int = 3

In [5]:
seq2.tail

res5: List[Int] = List(4, 5)

### Vector

Instead of lists, in which we typically access the head, and then the "next" element in the list, we can also use a `Vector`, where direct access to each element is possible through the index. 

Construction and other operations follow a ver similar pattern, but the internal implementation is different.

In [6]:
val vec1 = Vector(3,5,6)

val vec2 = 3 +: 5 +: 6 +: Vector.empty

vec1: Vector[Int] = Vector(3, 5, 6)
vec2: Vector[Int] = Vector(3, 5, 6)

### Set

Sets imply no duplicates in the collection. One implementation of it is a `HashSet`

In [7]:
val set1 = Set(4,6,7,4,3,5,6)

set1: Set[Int] = HashSet(5, 6, 7, 3, 4)

### Iterable

The iterable trait makes it possible to iterate over the element of a collection. Most of the colleciton classes will implement it. 

This allows to use methods such as `foreach`. We can implement our own iterable class.

In [8]:
class MyList[A](data:(A,A,A)) extends IterableOnce[A]:
  def foreach[U](f: A => U):Unit =
    for e <- this.data.toList do
      f(e)
 
  def iterator:Iterator[A] = ???

defined class MyList

Notice that the `foreach` return is always `Unit`, regardless of what function we pass as parameter. 

We can say it is not very "pure".

In [9]:
val mymy = new MyList[Char](('o','l','a'))
mymy.foreach(e=>println(e))

o
l
a


mymy: MyList[Char] = ammonite.$sess.cmd8$Helper$MyList@533a08b2

### Foreach

Looking more into `foreach`, let's see its signature:

```
  def foreach[U](f: A => U): Unit
```

One function we can pass and that is commonly used is a `match` `case`:

In [10]:
val seq = Seq(4,"2",7,"3",2)

seq.foreach(x => x match {
  case s:String => println(s"string $s")
  case i:Int => println(s"int $i")
})

int 4
string 2
int 7
string 3
int 2


seq: Seq[scala.collection.immutable.Seq[scala.Int | java.lang.String]] = List(
  4,
  "2",
  7,
  "3",
  2
)

In [ ]:
seq.foreach {
  case s:String => println(s"string $s")
  case i:Int => println(s"int $i")
}

### map

Looking into the signature of map, we can see that essentially there is a type transformation form `A` to `B`

```
class C[A]:
  def map[B](f: (A) => B): C[B]
```

In [11]:
case class Person(name:String, age:Int)

val persons = Seq(Person("amy",4),Person("jim",6),Person("tim",4))

defined class Person
persons: Seq[Person] = List(
  Person(name = "amy", age = 4),
  Person(name = "jim", age = 6),
  Person(name = "tim", age = 4)
)

In [ ]:
persons.map {
  case Person(name,4) => s"$name is in 1st Grade"
  case Person(name,_) => s"$name is in Another Grade"
}

In [12]:
val grades = (1 to 3).map(e => Seq(5,6,4))

grades: IndexedSeq[Seq[Int]] = Vector(
  List(5, 6, 4),
  List(5, 6, 4),
  List(5, 6, 4)
)

In [13]:
val grades = (1 to 3).map(e => Seq(e+1,e*2,e/2))

grades: IndexedSeq[Seq[Int]] = Vector(
  List(2, 2, 0),
  List(3, 4, 1),
  List(4, 6, 1)
)

### flatMap

Looking at the signature of flatmap, it looks similar to map, only that the return of the function parameter is often a collection `C`, so the parameter type of the class. 

```
class C[A]:
  def flatMap[B](f: A => Seq[B]): C[B]
  def map[B](f: (A) => B): C[B]
```

In [14]:
(1 to 5).flatMap(i => (1 to i))

res14: IndexedSeq[Int] = Vector(1, 1, 2, 1, 2, 3, 1, 2, 3, 4, 1, 2, 3, 4, 5)

In [15]:
(1 to 5).map(i => (1 to i))

res15: IndexedSeq[Inclusive] = Vector(
  Range(1),
  Range(1, 2),
  Range(1, 2, 3),
  Range(1, 2, 3, 4),
  Range(1, 2, 3, 4, 5)
)

Flatmap can be very useful when simplifying outputs of other compound types, like when we have collection of `Option`:

In [16]:
val grades = Seq(3,5,2,6,5,2)

def grading(grade:Int):Option[Int] = 
  if grade >= 4 then Some(grade) else None

grades.map(e=>grading(e))

grades: Seq[Int] = List(3, 5, 2, 6, 5, 2)
defined function grading
res16_2: Seq[Option[Int]] = List(
  None,
  Some(value = 5),
  None,
  Some(value = 6),
  Some(value = 5),
  None
)

In [17]:
grades.flatMap(e=>grading(e))

res17: Seq[Int] = List(5, 6, 5)

### fold and reduce and all the rest

Looking at the signature of `reduce`, we see that all types are esentially the same, all the pairwise elements to operate on, as well as the return.

```
def reduce[A1 >: A](op: (A1, A1) => A1): A1
```

In [18]:
val grades = Seq(3,5,2,6,5,2)

grades.reduce((x,y) => x+y)

grades: Seq[Int] = List(3, 5, 2, 6, 5, 2)
res18_1: Int = 23

In `foldLeft` the signature is very similar, but it introduces a "first element" `z`, and therefore the return type can actually change.

```
def foldLeft[B](z: B)(op: (B, A) => B): B
```

In [21]:
grades.foldLeft(4.9)((x,y) => x+y)

res21: Double = 27.9

In [22]:
grades.foldLeft(0)((x:Int,y:Int) => x+y)

res22: Int = 23

In [25]:
grades.foldLeft("3")((x,y) => x+y)

res25: String = "3352652"

So with `fold` in general we have further freedom to modify the types in different ways.

In [26]:
grades.foldLeft(0)((x,y) => x+y)

res26: Int = 23

In [27]:
grades.foldLeft(4)((x,y) => s"$x $y")

-- [E007] Type Mismatch Error: cmd28.sc:1:40 -----------------------------------
1 |val res28 = grades.foldLeft(4)((x,y) => s"$x $y")
  |                                        ^^^^^^^^
  |Found:    String
  |Required: Int
  |
  |One of the following imports might make progress towards fixing the problem:
  |
  |  import os.Path.stringPathValidated
  |  import os.PathChunk.stringPathChunkValidated
  |  import os.RelPath.stringRelPathValidated
  |  import os.SubPath.stringSubPathValidated
  |  import sourcecode.Text.generate
  |
  |
  | longer explanation available when compiling with `-explain`
Compilation Failed

In [28]:
grades.foldRight(Seq.empty){(x,y) => (x+0.5) +: y}

res28: Seq[Any] = List(3.5, 5.5, 2.5, 6.5, 5.5, 2.5)

## flatmap and Option

As we have seen `flatmap` works in two steps:

 - map on the input list to get an intermediate result, i.e., a “list of lists”
 - apply flatten to that intermediate result

For example:

In [29]:
val mapResult = List("foo", "bar").map(s=> s.split(""))

mapResult: List[Array[String]] = List(
  Array("f", "o", "o"),
  Array("b", "a", "r")
)

In [30]:
val flattenResult= mapResult.flatten

flattenResult: List[String] = List("f", "o", "o", "b", "a", "r")

In [31]:
 List("foo", "bar").flatMap(s => s.split(""))

res31: List[String] = List("f", "o", "o", "b", "a", "r")

## Options for wrapping results

We have seen cases where `Option` is useful as a wrapper for function return types.

We saw that `Option` is better than null values and exceptions, but those Options eventually coalesce at some point.

As an example consider this `makeInt` method:

In [32]:
def makeInt(s: String): Option[Int] = 
  try 
    Some(s.trim.toInt)
  catch 
    case e: Exception => None

defined function makeInt

In [33]:
val x = makeInt("3")

x: Option[Int] = Some(value = 3)

In [36]:
val y= makeInt("8")

y: Option[Int] = Some(value = 8)

Now what if I want to add the two `Int`s?

In [36]:
x + y

-- [E008] Not Found Error: cmd37.sc:1:14 ---------------------------------------
1 |val res37 = x + y
  |            ^^^
  |value + is not a member of Option[Int], but could be made available as an extension method.
  |
  |One of the following imports might make progress towards fixing the problem:
  |
  |  import scala.math.Fractional.Implicits.infixFractionalOps
  |  import scala.math.Integral.Implicits.infixIntegralOps
  |  import scala.math.Numeric.Implicits.infixNumericOps
  |  import sourcecode.Text.generate
  |
Compilation Failed

`Option` has no way to add the internal values packaged inside of it. We can write a code that does the sum if the values exist, or considers them 0 if not:

In [39]:
val x= makeInt("3")
val sum = x match 
  case None => 
    y match 
      case None => 0
      case Some(i) => i
  case Some(i) => 
    y match 
      case None => i
      case Some(j) => i + j

x: Option[Int] = Some(value = 3)
sum: Int = 11

This code is a bit ugly ust for doing the sum. Another alternative would be to use the `getOrElse` method of `Option`:

In [40]:
val sum = x.getOrElse(0) + y.getOrElse(0)

sum: Int = 11

Another way to access the "interior" of the `Option` is to use `map`:

In [41]:
x.map (a => a + 5)

res41: Option[Int] = Some(value = 8)

We can use map to add `x` and `y`:

In [44]:
val x  = makeInt("3")
x map( 
       a => y map (b => a + b) 
     )

x: Option[Int] = Some(value = 3)
res44_1: Option[Option[Int]] = Some(value = Some(value = 11))

Or in one line: 

In [45]:
x map( a => y map (b => a + b) )

res45: Option[Option[Int]] = Some(value = Some(value = 11))

Only problem is that we have a `Some` inside a `Some`. So we can use a `flatMap` instead:

In [46]:
x flatMap( a => y map (b => a + b) )

res46: Option[Int] = Some(value = 11)

## Using Option in `for` expressions

Now that we know that `Option` supports `map` and `flatMap`, this is an indication that we can also use it with `for` expressions:

In [47]:
val sum = for 
  x <- makeInt("1")
  y <- makeInt("2")
yield x + y

sum: Option[Int] = Some(value = 3)

It seems like a convenient way of operating with the `Option` values. In fact we can dd even more values:

In [48]:
val sum = for 
  a <- makeInt("1")
  b <- makeInt("2")
  c <- makeInt("3")
  d <- makeInt("4")
yield a + b + c + d

sum: Option[Int] = Some(value = 10)

This is much better than chaining together a whole bunch of `flatMap` and map functions, and it’s even easier to read than using a bunch of `getOrElse` calls.

In [50]:
val sum = makeInt("1").getOrElse(0) + makeInt("dfds").getOrElse(0) +
          makeInt("3").getOrElse(0) + makeInt("4").getOrElse(0)

sum: Int = 8

How about when soemthing goes wrong and one of the values is not an `Int`?

In [51]:
val sum = for 
  a <- makeInt("1")
  b <- makeInt("two")
  c <- makeInt("3")
  d <- makeInt("4")
yield a + b + c + d

sum: Option[Int] = None

When `makeInt` receives a `String` it can’t transform into an `Int`, these things happen:
 - `makeInt` returns a `None`
 - The `for` expression short-circuits at that point
 - sum is bound to `None`
 
The None result means that `makeInt` was given something it couldn’t convert to an `Int`

Internally, what happens in the compiler is that the codes gets transformed into something like:

```
val sum = makeInt("1").flatMap(
  ((a) => makeInt("2").flatMap(
    ((b) => makeInt("3").flatMap(
      ((c) => makeInt("4").map(
        ((d) => a.$plus(b).$plus(c).$plus(d)))))
      )
    )
  )
);
```

So internally the for expression is converted into multiple `flatMap` and a `map` in which the operation is executed.

## In summary

 - Pure functions don’t return `null` or throw exceptions, so they return `Option`, `Try`, etc.
 - Using `Option` naturally leads to certain code “patterns”
 - This eventually leads to the creation of values of the type `Option[Option[A]]`
 - `flatMap` works well with `Option[Option[A]]`, but it’s hard for humans to read
 - The Scala `for` expression is a human-readable replacement for a series of `flatMap` and `map` calls

### Also in collection function

All of the Scala collections classes have `map` and `flatMap` methods and work like this, including List, ArrayBuffer, Vector, etc., even the Map classes.

For example this code:

In [52]:
val xs = Seq(1,2,3)
val ys = Seq(100,200,300)

xs: Seq[Int] = List(1, 2, 3)
ys: Seq[Int] = List(100, 200, 300)

In [53]:
for
  x <- xs
  y <- ys
yield x + y

res53: Seq[Int] = List(101, 201, 301, 102, 202, 302, 103, 203, 303)

## Generalizing behaviors of `map` and `flatMap`

The key observation here is that: **any class that implements map and flatMap can be used in a for expression.**
 
Non-collection classes like Option, Try, Future, and many others can also be used in for expressions. 
These other classes tend to be “wrappers” of different types.

The Scala for expression is a sort of glue for getting pure functions to work together. 

For a class to be used with multiple generators in a for expression, that class must implement map and flatMap.
